<a href="https://colab.research.google.com/github/Nawaf-Alorabi/-Classification_Mini_Project/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anishdevedward/loan-approval-dataset",
    "loan_approval.csv"
)

/tmp/ipykernel_1961/751049816.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'loan-approval-dataset' dataset.


In [ ]:
df.head()

,name,city,income,credit_score,loan_amount,years_employed,points,loan_approved
0,Allison Hill,East Jill,113810,389,39698,27,50.0,False
1,Brandon Hall,New Jamesside,44592,729,15446,28,55.0,False
2,Rhonda Smith,Lake Roberto,33278,584,11189,13,45.0,False
3,Gabrielle Davis,West Melanieview,127196,344,48823,29,50.0,False
4,Valerie Gray,Mariastad,66048,496,47174,4,25.0,False


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2000 non-null   object 
 1   city            2000 non-null   object 
 2   income          2000 non-null   int64  
 3   credit_score    2000 non-null   int64  
 4   loan_amount     2000 non-null   int64  
 5   years_employed  2000 non-null   int64  
 6   points          2000 non-null   float64
 7   loan_approved   2000 non-null   bool   
dtypes: bool(1), float64(1), int64(4), object(2)
memory usage: 111.5+ KB


In [ ]:
df.shape

(2000, 8)

In [ ]:
df.describe()

,income,credit_score,loan_amount,years_employed,points
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,90585.977000,573.946000,25308.503000,20.441000,56.680000
std,34487.874907,160.564945,14207.320147,11.777813,18.638033
min,30053.000000,300.000000,1022.000000,0.000000,10.000000
25%,61296.250000,433.000000,12748.750000,10.000000,45.000000
50%,90387.500000,576.000000,25661.500000,21.000000,55.000000
75%,120099.750000,715.000000,37380.500000,31.000000,70.000000
max,149964.000000,850.000000,49999.000000,40.000000,100.000000


In [ ]:
df.isnull().sum()

,0
name,0
city,0
income,0
credit_score,0
loan_amount,0
years_employed,0
points,0
loan_approved,0


In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df = df.drop(["name", "city"], axis=1)

In [ ]:
df["loan_approved"] = df["loan_approved"].astype(int)

In [ ]:
df["loan_approved"].value_counts()

,count
loan_approved,
0,1121
1,879


In [ ]:
#Note: Before you begin training the model, ensure the data is balanced to prevent any bias towards the model!

#Pytorch model

In [ ]:
import torch
import numpy as np

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def fit(self, X, y, depth=0):
        # Stop conditions for recursion
        if depth >= self.max_depth or X.shape[0] < self.min_samples_split:
            self.tree = torch.mode(y).values.item()  # Assign majority class
            return

        # Find the best feature and threshold to split on
        best_split = self._best_split(X, y)
        if best_split is None:
            self.tree = torch.mode(y).values.item()
            return

        # Recursive split (Fixed)
        feature, threshold, left_indices, right_indices = best_split

        # Instantiate child trees
        left_child = DecisionTree(self.max_depth, self.min_samples_split)
        right_child = DecisionTree(self.max_depth, self.min_samples_split)

        # Fit child trees
        left_child.fit(X[left_indices], y[left_indices], depth + 1)
        right_child.fit(X[right_indices], y[right_indices], depth + 1)

        self.tree = {
            'feature': feature,
            'threshold': threshold,
            'left': left_child,   # Assign the fitted tree object, not the result of .fit()
            'right': right_child
        }

    def _gini_impurity(self, y):
        """Calculates the Gini Impurity for a set of labels."""
        if y.shape[0] == 0:
            return 0.0

        # Count the occurrences of each class in the current node
        _, counts = torch.unique(y, return_counts=True)

        # Calculate the probability of each class: p_i
        probabilities = counts.float() / y.shape[0]

        # Calculate Gini: 1 - sum(p_i^2)
        gini = 1.0 - torch.sum(probabilities ** 2).item()

        return gini

    def _gini_impurity(self, y):
        """Calculates the Gini Impurity for a set of PyTorch labels."""
        if y.shape[0] == 0:
            return 0.0

        # Count the occurrences of each class in the current node
        _, counts = torch.unique(y, return_counts=True)

        # Calculate the probability of each class: p_i
        probabilities = counts.float() / y.shape[0]

        # Calculate Gini: 1 - sum(p_i^2)
        gini = 1.0 - torch.sum(probabilities ** 2).item()

        return gini

    def _best_split(self, X, y):
        """Finds the best feature and threshold based on weighted Gini Impurity."""
        best_gini = float('inf')
        best_split = None
        n_samples, n_features = X.shape

        # --- RANDOM FOREST FEATURE SUBSETTING ---
        # Calculate the square root of total features (ensure at least 1 feature is picked)
        n_subset = max(1, int(n_features ** 0.5))

        # Pick a random subset of feature indices
        feature_indices = torch.randperm(n_features)[:n_subset].tolist()

        # Iterate ONLY through the randomly selected subset of features
        for feature_idx in feature_indices:

            # Get all unique values in this feature to test as potential split thresholds
            thresholds = torch.unique(X[:, feature_idx])

            for threshold in thresholds:
                # 1. Split the data based on the current threshold
                left_indices = torch.where(X[:, feature_idx] <= threshold)[0]
                right_indices = torch.where(X[:, feature_idx] > threshold)[0]

                # If a split leaves one side empty, it doesn't help us, so skip it
                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # 2. Calculate Gini for the left and right splits
                gini_left = self._gini_impurity(y[left_indices])
                gini_right = self._gini_impurity(y[right_indices])

                # 3. Calculate the Weighted Gini Impurity
                n_left = len(left_indices)
                n_right = len(right_indices)
                weighted_gini = (n_left / n_samples) * gini_left + (n_right / n_samples) * gini_right

                # 4. Check if this is the best split we've seen so far
                if weighted_gini < best_gini:
                    best_gini = weighted_gini
                    best_split = (feature_idx, threshold.item(), left_indices, right_indices)

        return best_split

    def predict(self, X):
        # Recursive prediction based on tree structure
        if isinstance(self.tree, dict):
            if X[:, self.tree['feature']] <= self.tree['threshold']:
                return self.tree['left'].predict(X)
            else:
                return self.tree['right'].predict(X)
        else:
            return self.tree  # Return leaf node prediction

In [ ]:
class RandomForest:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2):
        self.n_estimators = n_estimators
        self.trees = [DecisionTree(max_depth=max_depth, min_samples_split=min_samples_split) for _ in range(n_estimators)]

    def fit(self, X, y):
        for tree in self.trees:
            # Bootstrap sample
            indices = torch.randint(0, X.size(0), (X.size(0),))
            X_sample, y_sample = X[indices], y[indices]
            tree.fit(X_sample, y_sample)

    def predict(self, X):
        # Create an empty tensor to hold predictions for the whole batch
        predictions = torch.zeros(X.shape[0])

        # Predict one row at a time
        for i in range(X.shape[0]):
            predictions[i] = self._predict_single(X[i], self.tree)

        return predictions

    def _predict_single(self, x, node):
        # Base case: if it's a value (not a dict), we've hit a leaf
        if not isinstance(node, dict):
            return node

        # Traverse the tree
        if x[node['feature']] <= node['threshold']:
            # Notice we pass .tree because 'left' is a DecisionTree object
            return self._predict_single(x, node['left'].tree)
        else:
            return self._predict_single(x, node['right'].tree)